# Tech Job Trends — Stage 1: Build ChromaDB Index

This notebook prepares the job-posting dataset and builds a persistent ChromaDB vector index.

**Pipeline:** CSV → cleaning → chunking → embeddings → ChromaDB


## Configuration

### Colab
Upload `postings.csv` to the runtime, then set `DATA_PATH` to `/content/postings.csv`.

### Local
Use a local path such as `data/postings.csv`.

The generated ChromaDB directory is intentionally kept outside GitHub by `.gitignore`.


In [ ]:
from pathlib import Path
import uuid

import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_PATH = Path("/content/postings.csv")  # Change for local use
CHROMA_DB_PATH = Path("./chroma_db")
COLLECTION_NAME = "tech_jobs"
MAX_JOBS = 10_000
BATCH_SIZE = 100


In [ ]:
def clean_text(text) -> str:
    """Normalize a job description for indexing."""
    if pd.isna(text):
        return ""
    return str(text).replace("\n", " ").strip()


def load_job_postings(path: Path, max_jobs: int = MAX_JOBS) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {path}. Upload postings.csv or update DATA_PATH."
        )

    df = pd.read_csv(path).head(max_jobs)
    print(f"Loaded {len(df):,} job postings.")
    return df


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)


In [ ]:
embedding_fn = embedding_functions.DefaultEmbeddingFunction()
client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

print(f"ChromaDB collection ready: {COLLECTION_NAME}")


In [ ]:
def index_jobs(df: pd.DataFrame, batch_size: int = BATCH_SIZE) -> int:
    documents_batch = []
    metadatas_batch = []
    ids_batch = []
    total_chunks = 0

    for idx, row in df.iterrows():
        job_title = row.get("title", "Unknown Title")
        company = row.get("company", "Unknown Company")
        description = clean_text(row.get("description", ""))

        if not description:
            continue

        parent_id = f"job_{uuid.uuid4().hex[:8]}"
        chunks = text_splitter.split_text(description)

        for chunk_index, chunk in enumerate(chunks):
            documents_batch.append(chunk)
            metadatas_batch.append({
                "parent_id": parent_id,
                "job_title": str(job_title),
                "company": str(company),
                "chunk_index": chunk_index,
                "total_chunks": len(chunks),
            })
            ids_batch.append(f"{parent_id}_chunk_{chunk_index}")

            if len(documents_batch) >= batch_size:
                collection.add(
                    documents=documents_batch,
                    metadatas=metadatas_batch,
                    ids=ids_batch,
                )
                total_chunks += len(documents_batch)
                documents_batch, metadatas_batch, ids_batch = [], [], []

        print(f"Indexed job {idx + 1}/{len(df)} → {len(chunks)} chunks")

    if documents_batch:
        collection.add(
            documents=documents_batch,
            metadatas=metadatas_batch,
            ids=ids_batch,
        )
        total_chunks += len(documents_batch)

    return total_chunks


In [ ]:
df = load_job_postings(DATA_PATH)
total_chunks = index_jobs(df)
print(f"\nDone — stored {total_chunks:,} chunks in ChromaDB.")


## Result

The persistent vector store is now available at `./chroma_db`.
The next notebook loads this collection and performs semantic retrieval before generating an answer with Gemini.
